# 01 Data Audit: Mining & Processing Pipeline

This notebook walks through the full Phase 1 pipeline for the **How Fast Is Fashion** project:

1. **Setup** -- imports, config, paths
2. **Mining Configuration** -- load and inspect mining config
3. **Run Mining Pipeline** -- mine archived product images from the Wayback Machine
4. **Inspect Raw Data** -- audit raw images and metadata records
5. **Cleaning Summary** -- compare raw vs clean counts per month
6. **Normalization** -- LLM-based metadata normalization
7. **TF-IDF Keyword Extraction** -- extract distinctive terms per year
8. **Trend Mapping** -- map features to named trends via taxonomy
9. **Trend State Analysis** -- classify features as rising/falling/stable
10. **Export** -- save final parquet files

Run cells top-to-bottom once data exists (or start from Section 3 to mine fresh data).

## 1. Setup

Import project modules and standard libraries, initialize paths.

In [ ]:
import sys
from pathlib import Path

# Ensure the project root is on sys.path so we can import fashion_forensics
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import json

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from PIL import Image

from fashion_forensics.config import (
    DATA_DIR,
    PROJECT_ROOT,
    load_yaml_config,
)
from fashion_forensics.mining.miner import (
    CLEANED_DIR,
    RAW_IMAGES_DIR,
    RECORDS_DIR,
    STATE_DIR,
    FashionMiner,
    MiningConfig,
)
from fashion_forensics.nlp.mapper import load_taxonomy, map_features_to_trends
from fashion_forensics.nlp.trend_engine import compute_trend_states
from fashion_forensics.normalization.normalizer import (
    OUTPUTS_DIR as NORM_OUTPUTS_DIR,
)
from fashion_forensics.normalization.normalizer import (
    REVIEW_DIR as NORM_REVIEW_DIR,
)
from fashion_forensics.normalization.normalizer import (
    normalize_month,
)

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 60)

print(f"Project root : {PROJECT_ROOT}")
print(f"Data dir     : {DATA_DIR}")
print(f"Raw images   : {RAW_IMAGES_DIR}")
print(f"Records      : {RECORDS_DIR}")
print(f"Cleaned      : {CLEANED_DIR}")

## 2. Mining Configuration

Load the YAML mining config and display key parameters. The config controls:
- **target_url** -- which retail page to mine from the Wayback Machine
- **date range** -- start/end months for snapshot queries
- **sampling_strategy** -- `every_month`, `quarterly`, or `random_months`
- **images_per_month** -- target number of clean images per month
- **URL signals** -- positive/negative patterns to filter product images from page assets

In [ ]:
# Load raw YAML config
raw_config = load_yaml_config("mining")

# Display as a formatted table
config_df = pd.DataFrame(
    [(k, v) for k, v in raw_config.items()],
    columns=["Parameter", "Value"],
)
display(config_df)

# Parse into MiningConfig for downstream use
mining_cfg = MiningConfig(raw_config)
print(f"\nSampling strategy : {mining_cfg.sampling_strategy}")
print(f"Date range        : {mining_cfg.start_date} -> {mining_cfg.end_date}")
print(f"Target clean/month: {mining_cfg.target_clean_per_month}")
print(f"Min image bytes   : {mining_cfg.min_image_bytes:,}")

In [ ]:
# Preview which months will be mined under each strategy
miner_preview = FashionMiner(mining_cfg)
months = miner_preview.generate_months()

print(f"Months to mine ({mining_cfg.sampling_strategy}): {len(months)}")
for m in months:
    print(f"  {m[:4]}-{m[4:]}")

## 3. Run Mining Pipeline

Instantiate `FashionMiner` and execute the full run. For each month, the miner:
1. Queries the Wayback Machine CDX API for archived snapshots
2. Parses the archived page for product image URLs
3. Downloads and deduplicates images into `raw_images/YYYY-MM/`
4. Writes append-only JSONL metadata to `records/`
5. Applies cleaning rules and writes passing records to `cleaned/`
6. Updates per-month state with adaptive keep rate

**Note:** This cell makes live HTTP requests to the Wayback Machine. It is safe to re-run -- the miner resumes from saved state and skips already-completed months.

In [ ]:
miner = FashionMiner(mining_cfg)
results = miner.run()

In [ ]:
# Summarize mining results
results_df = pd.DataFrame.from_dict(results, orient="index")
results_df.index.name = "month"
results_df = results_df.reset_index()

print(f"Total months processed: {len(results_df)}")
print(f"Total raw added       : {results_df['raw_added'].sum()}")
print(f"Total clean added     : {results_df['clean_added'].sum()}")
display(results_df)

## 4. Inspect Raw Data

Count images per month folder, display sample images, and show sample JSONL records.

In [ ]:
# Count images per month folder
month_folders = sorted(RAW_IMAGES_DIR.iterdir()) if RAW_IMAGES_DIR.exists() else []

counts = {}
for folder in month_folders:
    if folder.is_dir():
        images = (
            list(folder.glob("*.jpg")) + list(folder.glob("*.png")) + list(folder.glob("*.webp"))
        )
        counts[folder.name] = len(images)

if counts:
    counts_df = pd.DataFrame(list(counts.items()), columns=["Month", "Image Count"])
    display(counts_df)

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.bar(counts_df["Month"], counts_df["Image Count"], color="steelblue")
    ax.set_xlabel("Month")
    ax.set_ylabel("Number of Raw Images")
    ax.set_title("Raw Images per Month")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("No raw image folders found yet. Run the mining pipeline first.")

In [ ]:
# Display sample images from the first available month folder
if month_folders:
    sample_folder = month_folders[0]
    sample_images = sorted(sample_folder.glob("*.jpg"))[:6]
    if not sample_images:
        sample_images = sorted(sample_folder.glob("*.png"))[:6]

    if sample_images:
        n = len(sample_images)
        fig, axes = plt.subplots(1, n, figsize=(3 * n, 4))
        if n == 1:
            axes = [axes]
        for ax, img_path in zip(axes, sample_images):
            img = Image.open(img_path)
            ax.imshow(img)
            ax.set_title(img_path.name, fontsize=8)
            ax.axis("off")
        fig.suptitle(f"Sample images from {sample_folder.name}", fontsize=12)
        plt.tight_layout()
        plt.show()
    else:
        print(f"No images found in {sample_folder}")
else:
    print("No month folders available.")

In [ ]:
# Show sample records from the first available JSONL file
record_files = sorted(RECORDS_DIR.glob("*_records.jsonl"))

if record_files:
    sample_file = record_files[0]
    lines = sample_file.read_text().strip().split("\n")
    sample_records = [json.loads(line) for line in lines[:5]]
    print(f"Sample records from {sample_file.name} ({len(lines)} total records):\n")
    display(pd.DataFrame(sample_records))
else:
    print("No record files found yet.")

## 5. Cleaning Summary

Compare raw vs clean record counts per month. The miner applies cleaning rules (e.g., minimum image size) and tracks an adaptive keep rate in the per-month state files.

In [ ]:
# Build a summary from state files
state_files = sorted(STATE_DIR.glob("*_state.json"))

if state_files:
    rows = []
    for sf in state_files:
        data = json.loads(sf.read_text())
        rows.append(
            {
                "month": data.get("month", sf.stem.replace("_state", "")),
                "raw_count": data.get("raw_count", 0),
                "clean_count": data.get("clean_count", 0),
                "keep_rate": round(data.get("estimated_keep_rate", 0), 3),
                "urls_attempted": len(data.get("attempted_urls", [])),
            }
        )

    summary_df = pd.DataFrame(rows)
    summary_df["drop_count"] = summary_df["raw_count"] - summary_df["clean_count"]
    display(summary_df)

    # Bar chart: raw vs clean per month
    fig, ax = plt.subplots(figsize=(12, 5))
    x = range(len(summary_df))
    width = 0.35
    ax.bar([i - width / 2 for i in x], summary_df["raw_count"], width, label="Raw", color="salmon")
    ax.bar(
        [i + width / 2 for i in x],
        summary_df["clean_count"],
        width,
        label="Clean",
        color="seagreen",
    )
    ax.set_xticks(list(x))
    ax.set_xticklabels(summary_df["month"], rotation=45, ha="right")
    ax.set_ylabel("Count")
    ax.set_title("Raw vs Clean Images per Month")
    ax.legend()
    plt.tight_layout()
    plt.show()

    # Keep rate over time
    fig, ax = plt.subplots(figsize=(12, 3))
    ax.plot(summary_df["month"], summary_df["keep_rate"], marker="o", color="darkorange")
    ax.set_ylabel("Keep Rate")
    ax.set_title("Adaptive Keep Rate by Month")
    ax.set_ylim(0, 1)
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("No state files found. Run the mining pipeline first.")

## 6. Normalization

Run the LLM-based normalizer on cleaned records. Each clean record is sent to Gemini
to produce standardized metadata (category, color, material, silhouette, pattern, details).
Results are written to `normalization/outputs/` as JSONL and `normalization/review/` as CSV
for human inspection. All LLM calls are traced through Langfuse.

In [ ]:
# Run normalization for each month that has cleaned data
clean_files = sorted(CLEANED_DIR.glob("*_clean.jsonl"))

if clean_files:
    norm_results = {}
    for cf in clean_files:
        month_str = cf.stem.replace("_clean", "")
        count = normalize_month(month_str)
        norm_results[month_str] = count
        print(f"  {month_str}: {count} records normalized")

    print(f"\nTotal normalized: {sum(norm_results.values())}")
else:
    print("No cleaned data files found. Run the mining pipeline first.")

In [ ]:
# Display sample normalized records
norm_files = sorted(NORM_OUTPUTS_DIR.glob("*_normalized.jsonl"))

if norm_files:
    sample_norm = norm_files[0]
    lines = sample_norm.read_text().strip().split("\n")
    sample_records = [json.loads(line) for line in lines[:5]]
    print(f"Sample normalized records from {sample_norm.name} ({len(lines)} total):\n")
    display(pd.json_normalize(sample_records))
else:
    print("No normalized output files found.")

In [ ]:
# Display review CSVs
review_files = sorted(NORM_REVIEW_DIR.glob("*_review.csv"))

if review_files:
    sample_review = review_files[0]
    review_df = pd.read_csv(sample_review)
    print(f"Review CSV: {sample_review.name} ({len(review_df)} rows)\n")
    display(review_df.head(10))
else:
    print("No review CSVs found.")

## 7. TF-IDF Keyword Extraction

Use `sklearn.feature_extraction.text.TfidfVectorizer` on normalized product text grouped
by year. This surfaces terms that are **distinctive** to each year (not just frequent).
High TF-IDF terms are features that characterize a year's product assortment relative
to other years.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Load all normalized records and build text per item
all_norm_records = []
for nf in sorted(NORM_OUTPUTS_DIR.glob("*_normalized.jsonl")):
    for line in nf.read_text().strip().split("\n"):
        if line:
            rec = json.loads(line)
            # Derive year from the file name (YYYYMM_normalized.jsonl)
            month_str = nf.stem.replace("_normalized", "")
            rec["year"] = int(month_str[:4])
            all_norm_records.append(rec)

if all_norm_records:
    norm_df = pd.json_normalize(all_norm_records)

    # Build a text representation per item from normalized fields
    text_fields = [
        "category",
        "subcategory",
        "color_primary",
        "color_secondary",
        "material_primary",
        "material_secondary",
        "silhouette",
        "pattern",
    ]

    def build_text(row):
        parts = []
        for f in text_fields:
            val = row.get(f)
            if pd.notna(val) and val and str(val).lower() != "null":
                parts.append(str(val).lower())
        details = row.get("details", [])
        if isinstance(details, list):
            parts.extend([str(d).lower() for d in details])
        return " ".join(parts)

    norm_df["item_text"] = norm_df.apply(build_text, axis=1)

    # Group text by year -- each year becomes one "document" for TF-IDF
    years = sorted(norm_df["year"].unique())
    year_docs = {y: " ".join(norm_df[norm_df["year"] == y]["item_text"].tolist()) for y in years}

    vectorizer = TfidfVectorizer(max_features=200, stop_words="english", ngram_range=(1, 2))
    tfidf_matrix = vectorizer.fit_transform(year_docs.values())
    feature_names = vectorizer.get_feature_names_out()

    # Extract top terms per year
    top_n = 15
    tfidf_results = {}
    for i, year in enumerate(years):
        scores = tfidf_matrix[i].toarray().flatten()
        top_indices = scores.argsort()[-top_n:][::-1]
        tfidf_results[year] = [(feature_names[j], round(scores[j], 4)) for j in top_indices]

    # Display as table
    for year, terms in tfidf_results.items():
        print(f"\n--- Top {top_n} TF-IDF terms for {year} ---")
        display(pd.DataFrame(terms, columns=["Term", "TF-IDF Score"]))
else:
    print("No normalized records found. Run normalization first.")

## 8. Trend Mapping

Load the trend taxonomy (`trend_taxonomy_2024_2026.json`) and use `map_features_to_trends`
to match each year's distinctive TF-IDF features to named fashion trends (e.g., "quiet_luxury",
"coquette", "office_siren"). This step uses Gemini to perform the mapping, constrained to
only use trend names from the taxonomy.

In [ ]:
# Load the trend taxonomy
taxonomy = load_taxonomy()
print(f"Taxonomy contains {len(taxonomy)} trends:\n")
for name, info in taxonomy.items():
    active = ", ".join(str(y) for y in info.get("years_active", []))
    print(f"  {name}: {info['description'][:60]}... (active: {active})")

In [ ]:
# Map TF-IDF features to trends for each year
if tfidf_results:
    trend_mappings = {}
    for year, terms in tfidf_results.items():
        features = [term for term, score in terms]
        print(f"\nMapping {len(features)} features for {year}...")
        mapping = map_features_to_trends(features, year, taxonomy=taxonomy)
        trend_mappings[year] = mapping

        # Display results
        for trend_name, matched_features in mapping.items():
            print(f"  {trend_name}: {', '.join(matched_features)}")
else:
    print("No TF-IDF results available. Run Section 7 first.")

## 9. Trend State Analysis

Compute year-over-year trend states using `compute_trend_states`. Each feature is classified
as **rising**, **falling**, **stable**, **emerging**, or **archived** based on its YoY share delta.
This drives the narrative analysis downstream.

In [ ]:
# Build yearly metrics from normalized data (feature counts and shares per year)
if all_norm_records:
    # Count occurrences of each feature (category, color, material, pattern, silhouette) per year
    feature_rows = []
    for _, row in norm_df.iterrows():
        year = row.get("year")
        for field in [
            "category",
            "subcategory",
            "color_primary",
            "material_primary",
            "silhouette",
            "pattern",
        ]:
            val = row.get(field)
            if pd.notna(val) and val and str(val).lower() != "null":
                feature_rows.append({"feature": str(val).lower(), "year": year})

    feature_df = pd.DataFrame(feature_rows)
    yearly_counts = feature_df.groupby(["feature", "year"]).size().reset_index(name="count")

    # Compute share within each year
    year_totals = yearly_counts.groupby("year")["count"].transform("sum")
    yearly_counts["share"] = yearly_counts["count"] / year_totals

    print(f"Feature-year combinations: {len(yearly_counts)}")
    display(yearly_counts.head(10))
else:
    print("No normalized data available.")

In [ ]:
# Compute trend states
if all_norm_records:
    trend_states_df = compute_trend_states(yearly_counts)

    # Show rising features
    rising = trend_states_df[trend_states_df["state"] == "rising"].sort_values(
        "yoy_delta", ascending=False
    )
    print(f"RISING features ({len(rising)}):")
    if len(rising) > 0:
        display(rising.head(10))

    # Show falling features
    falling = trend_states_df[trend_states_df["state"] == "falling"].sort_values("yoy_delta")
    print(f"\nFALLING features ({len(falling)}):")
    if len(falling) > 0:
        display(falling.head(10))

    # Show stable features
    stable = trend_states_df[trend_states_df["state"] == "stable"]
    print(f"\nSTABLE features ({len(stable)}):")
    if len(stable) > 0:
        display(stable.head(10))

    # State distribution chart
    state_counts = trend_states_df["state"].value_counts()
    fig, ax = plt.subplots(figsize=(8, 4))
    colors = {
        "rising": "forestgreen",
        "falling": "firebrick",
        "stable": "steelblue",
        "emerging": "gold",
        "archived": "gray",
        "seasonal": "darkorange",
    }
    bar_colors = [colors.get(s, "steelblue") for s in state_counts.index]
    state_counts.plot(kind="bar", ax=ax, color=bar_colors)
    ax.set_title("Feature Trend State Distribution")
    ax.set_ylabel("Count")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

## 10. Export

Save the final processed datasets as parquet files:
- `item_features.parquet` -- normalized item-level features with year
- `trend_metrics_yearly.parquet` -- yearly feature counts, shares, and trend states

In [ ]:
EXPORT_DIR = DATA_DIR / "01_data_audit" / "exports"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

if all_norm_records:
    # Export item-level features
    item_features_path = EXPORT_DIR / "item_features.parquet"
    export_cols = [
        "record_id",
        "year",
        "category",
        "subcategory",
        "color_primary",
        "color_secondary",
        "material_primary",
        "material_secondary",
        "silhouette",
        "pattern",
        "confidence",
    ]
    available_cols = [c for c in export_cols if c in norm_df.columns]
    norm_df[available_cols].to_parquet(item_features_path, index=False)
    print(f"Saved item_features.parquet: {len(norm_df)} rows -> {item_features_path}")

    # Export trend metrics
    trend_metrics_path = EXPORT_DIR / "trend_metrics_yearly.parquet"
    trend_states_df.to_parquet(trend_metrics_path, index=False)
    print(
        f"Saved trend_metrics_yearly.parquet: {len(trend_states_df)} rows -> {trend_metrics_path}"
    )
else:
    print("No data to export. Run the full pipeline first.")

---

**Pipeline complete.** The exported parquet files in `data/01_data_audit/exports/` are ready for downstream analysis in Phase 2.